In [91]:
# BnF: le constitutionnel (1819) and le journal des débats politiques et littéraires (1814-1819)

## Extract newspapers from BnF zips + preprocess

In [92]:
import zipfile
import json
import pandas as pd
from langchain_text_splitters import RecursiveCharacterTextSplitter
from datasets import Dataset

In [93]:
ds_path = "datasets_all/bnfnewspapers-subset"

In [94]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    model_name="gpt-2", chunk_size=300, chunk_overlap=100
)

In [95]:
raw_filepaths = ["raw/french_newspapers/le_constitutionnel.zip", "raw/french_newspapers/le_journal_des_debats_politiques_et_litteraires.zip"]

In [96]:
years = [1805, 1806, 1807, 1808, 1809, 1810, 1811, 1812, 1813, 1814, 1815, 1816, 1817, 1818, 1819]
min_ocr_conf = 0.7

In [97]:
rows = []

for filepath in raw_filepaths:
    with zipfile.ZipFile(filepath, "r") as zf:
        for file_name in zf.namelist():
            if "fulltext" in file_name and int(file_name.split("/")[0]) in years:
                # read json file
                with zf.open(file_name) as f:
                    json_content = f.read()
                    jsonfile = json.loads(json_content.decode("utf-8"))
                    
                    ocr_conf = float(jsonfile['format'][0].split(' ')[2].replace(',','.'))
                    year = jsonfile['date'][0].split('-')[0]
                    if "French" in jsonfile['language'] and ocr_conf>=min_ocr_conf:
                        for x in jsonfile["contentAsText"]:
                            paras = text_splitter.split_text(str(x.replace('\n', ' ')))
                            for p in paras:
                                rows.append([p, year, jsonfile['title'][0]])

In [98]:
newspaper_contents = pd.DataFrame(rows, columns=["Text", "Year", "Newspaper"])
ds = Dataset.from_pandas(newspaper_contents)
ds.save_to_disk(ds_path)

Saving the dataset (0/1 shards):   0%|          | 0/152523 [00:00<?, ? examples/s]

In [102]:
ds[0]

{'Text': 'vendredi 1" Avril «s «4. JOURNAL DES DEBATS, AVIS. MM. le# Smiscripleurs des départeinens, dont l\'abonnement finit. le i S de ce mois, sont priés\'de le l\'aire renouveler pour ne pas éprou ver de retard. Le prix de Vabonnement au JOURNAL DES DEBATS, ci devant de PEMPIRE, de quinze fr. pour trais mois, de trente ir.pour six mois , et de soixante ir. pour t année. Les lettres, paquets et argent, doirent être adressés, franc de port, au bureau dudit Journal, rue des Prêtres Saint- Germain- F Aux errois , 17 , et les effets passés à Fordre-du caissier. On est prié de joindre à toutes- les réclamations , ebangemens d\'adresses ainsi que pour les réabonnemens, la dernière adresse imprimée que l\'on a reçue avec le Journal ; ou sera servi plus promptement. ANGLETERRE. Londres , 20 mars.',
 'Year': '1814',
 'Newspaper': 'Le Journal des Débats politiques et littéraires'}

In [103]:
from data import BNFNewspaper

ds_cleaned = BNFNewspaper()

Cleaning data since cleaned version not found


Saving the dataset (0/1 shards):   0%|          | 0/152523 [00:00<?, ? examples/s]